# GenAI ROI Analysis - Part 2: Data Cleaning & Preparation

**Author:** [Your Name]  
**Date:** March 2025

---

## Overview

This notebook cleans and prepares the survey data for analysis:
1. Filter relevant columns
2. Create derived metrics
3. Handle missing values
4. Calculate ROI fields

---

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported!")

## 2. Load Raw Data

In [ ]:
# Load data
df = pd.read_csv('../data/raw/survey_results_public.csv')
print(f"Original shape: {df.shape}")

## 3. Filter Columns

In [ ]:
# Columns to keep
COLUMNS_TO_KEEP = [
    'AISelect', 'AISent', 'AIBen', 'AIAcc', 'AIComplex', 
    'AIThreat', 'AIEthics', 'AIChallenges',
    'MainBranch', 'DevType', 'OrgSize', 'YearsCode', 'YearsCodePro',
    'Country', 'CompTotal', 'Currency', 'Industry', 'Employment',
    'RemoteWork', 'EdLevel', 'Age',
    'TimeSearching', 'TimeAnswering', 'JobSat',
    'PurchaseInfluence', 'ICorPM', 'WorkExp'
]

cols = [c for c in COLUMNS_TO_KEEP if c in df.columns]
df = df[cols]

# Filter to those who answered AI question
df = df[df['AISelect'].notna()]
print(f"Filtered shape: {df.shape}")

## 4. Create AI Status Fields

In [ ]:
# AI Status
df['AI_Status'] = df['AISelect'].map({
    'Yes': 'Using AI',
    'No, but I plan to soon': 'Planning to Use',
    "No, and I don't plan to": 'Not Using'
})

# Binary flag
df['Is_AI_User'] = (df['AISelect'] == 'Yes').astype(int)

print("AI Status:")
print(df['AI_Status'].value_counts())

## 5. Create Sentiment Scores

In [ ]:
# Sentiment Score (1-5)
SENTIMENT_MAP = {
    'Very unfavorable': 1,
    'Unfavorable': 2,
    'Indifferent': 3,
    'Unsure': 3,
    'Favorable': 4,
    'Very favorable': 5
}

df['Sentiment_Score'] = df['AISent'].map(SENTIMENT_MAP)

# Sentiment Category
df['Sentiment_Category'] = df['AISent'].map({
    'Very unfavorable': 'Negative',
    'Unfavorable': 'Negative',
    'Indifferent': 'Neutral',
    'Unsure': 'Neutral',
    'Favorable': 'Positive',
    'Very favorable': 'Positive'
})

print("Sentiment Categories:")
print(df['Sentiment_Category'].value_counts())

## 6. Create Trust & Complexity Scores

In [ ]:
# Trust Score
TRUST_MAP = {
    'Highly distrust': 1,
    'Somewhat distrust': 2,
    'Neither trust nor distrust': 3,
    'Somewhat trust': 4,
    'Highly trust': 5
}
df['Trust_Score'] = df['AIAcc'].map(TRUST_MAP)

# Complexity Score
COMPLEXITY_MAP = {
    'Very poor at handling complex tasks': 1,
    'Bad at handling complex tasks': 2,
    'Neither good or bad at handling complex tasks': 3,
    'Good, but not great at handling complex tasks': 4,
    'Very well at handling complex tasks': 5
}
df['Complexity_Score'] = df['AIComplex'].map(COMPLEXITY_MAP)

print(f"Trust Score: {df['Trust_Score'].mean():.2f} avg")
print(f"Complexity Score: {df['Complexity_Score'].mean():.2f} avg")

## 7. Standardize Company Size

In [ ]:
# Company Size Categories
SIZE_MAP = {
    'Just me - I am a freelancer, sole proprietor, etc.': 'Solo/Freelancer',
    '2 to 9 employees': 'Micro (2-9)',
    '10 to 19 employees': 'Small (10-19)',
    '20 to 99 employees': 'Small-Mid (20-99)',
    '100 to 499 employees': 'Mid (100-499)',
    '500 to 999 employees': 'Mid-Large (500-999)',
    '1,000 to 4,999 employees': 'Large (1K-5K)',
    '5,000 to 9,999 employees': 'Large (5K-10K)',
    '10,000 or more employees': 'Enterprise (10K+)',
    "I don't know": 'Unknown'
}

df['Company_Size_Category'] = df['OrgSize'].map(SIZE_MAP)

# Sort order
SIZE_ORDER = {
    'Solo/Freelancer': 1, 'Micro (2-9)': 2, 'Small (10-19)': 3,
    'Small-Mid (20-99)': 4, 'Mid (100-499)': 5, 'Mid-Large (500-999)': 6,
    'Large (1K-5K)': 7, 'Large (5K-10K)': 8, 'Enterprise (10K+)': 9,
    'Unknown': 0
}
df['Company_Size_Order'] = df['Company_Size_Category'].map(SIZE_ORDER)

print("Company Size Categories:")
print(df['Company_Size_Category'].value_counts())

## 8. Clean Experience Levels

In [ ]:
def clean_years(val):
    if pd.isna(val):
        return np.nan
    if val == 'Less than 1 year':
        return 0.5
    if val == 'More than 50 years':
        return 50
    try:
        return float(val)
    except:
        return np.nan

df['Years_Pro_Clean'] = df['YearsCodePro'].apply(clean_years)

# Experience Level
def get_exp_level(years):
    if pd.isna(years):
        return 'Unknown'
    if years < 2:
        return 'Junior (0-2)'
    elif years < 5:
        return 'Mid (2-5)'
    elif years < 10:
        return 'Senior (5-10)'
    else:
        return 'Expert (10+)'

df['Experience_Level'] = df['Years_Pro_Clean'].apply(get_exp_level)

print("Experience Levels:")
print(df['Experience_Level'].value_counts())

## 9. Simplify Job Roles

In [ ]:
def simplify_role(role):
    if pd.isna(role):
        return 'Unknown'
    role = str(role).lower()
    if 'full-stack' in role:
        return 'Full-Stack Developer'
    elif 'back-end' in role:
        return 'Back-End Developer'
    elif 'front-end' in role:
        return 'Front-End Developer'
    elif 'mobile' in role:
        return 'Mobile Developer'
    elif 'data scientist' in role or 'machine learning' in role:
        return 'Data Scientist/ML'
    elif 'data engineer' in role:
        return 'Data Engineer'
    elif 'devops' in role or 'sre' in role:
        return 'DevOps/SRE'
    elif 'manager' in role or 'executive' in role:
        return 'Manager/Executive'
    elif 'student' in role:
        return 'Student'
    elif 'academic' in role or 'research' in role:
        return 'Researcher/Academic'
    else:
        return 'Other Developer'

df['Role_Category'] = df['DevType'].apply(simplify_role)

print("Role Categories:")
print(df['Role_Category'].value_counts())

## 10. Extract Benefits & Challenges

In [ ]:
# Benefits
BENEFITS = ['Increase productivity', 'Speed up learning', 'Greater efficiency',
            'Improve accuracy in coding', 'Make workload more manageable', 'Improve collaboration']

for benefit in BENEFITS:
    col = 'Ben_' + benefit.replace(' ', '_')[:20]
    df[col] = df['AIBen'].fillna('').str.contains(benefit, case=False).astype(int)

df['Benefits_Count'] = df['AIBen'].fillna('').apply(lambda x: len(x.split(';')) if x else 0)

# Challenges
df['Challenge_Trust'] = df['AIChallenges'].fillna('').str.contains('trust', case=False).astype(int)
df['Challenge_Context'] = df['AIChallenges'].fillna('').str.contains('context|codebase', case=False).astype(int)
df['Challenge_Security'] = df['AIChallenges'].fillna('').str.contains('security', case=False).astype(int)
df['Challenge_Training'] = df['AIChallenges'].fillna('').str.contains('training', case=False).astype(int)
df['Challenge_Adoption'] = df['AIChallenges'].fillna('').str.contains('Not everyone', case=False).astype(int)

df['Challenges_Count'] = df['AIChallenges'].fillna('').apply(lambda x: len(x.split(';')) if x else 0)

print(f"Avg benefits selected: {df[df['Is_AI_User']==1]['Benefits_Count'].mean():.1f}")
print(f"Avg challenges cited: {df[df['Is_AI_User']==1]['Challenges_Count'].mean():.1f}")

## 11. Clean Compensation & Calculate ROI

In [ ]:
# Clean compensation
df['Comp_Clean'] = df['CompTotal']
df.loc[df['Comp_Clean'] > 1000000, 'Comp_Clean'] = np.nan
df.loc[df['Comp_Clean'] < 1000, 'Comp_Clean'] = np.nan

# Hourly rate (2000 hours/year)
df['Hourly_Rate_Est'] = df['Comp_Clean'] / 2000

# ROI Calculations
TOOL_COST = 240  # $20/month

df['Hours_Saved_Weekly_Low'] = 2
df['Hours_Saved_Weekly_Mid'] = 5
df['Hours_Saved_Weekly_High'] = 8

df['Value_Annual_Low'] = df['Hours_Saved_Weekly_Low'] * df['Hourly_Rate_Est'] * 50
df['Value_Annual_Mid'] = df['Hours_Saved_Weekly_Mid'] * df['Hourly_Rate_Est'] * 50
df['Value_Annual_High'] = df['Hours_Saved_Weekly_High'] * df['Hourly_Rate_Est'] * 50

df['ROI_Low'] = ((df['Value_Annual_Low'] - TOOL_COST) / TOOL_COST * 100).round(1)
df['ROI_Mid'] = ((df['Value_Annual_Mid'] - TOOL_COST) / TOOL_COST * 100).round(1)
df['ROI_High'] = ((df['Value_Annual_High'] - TOOL_COST) / TOOL_COST * 100).round(1)

print(f"Median ROI (Mid scenario): {df['ROI_Mid'].median():,.0f}%")

## 12. Add Regions

In [ ]:
def get_region(country):
    if pd.isna(country):
        return 'Unknown'
    country = str(country)
    if country in ['United States of America', 'Canada']:
        return 'North America'
    elif country in ['United Kingdom of Great Britain and Northern Ireland', 'Germany', 'France', 
                     'Netherlands', 'Sweden', 'Poland', 'Italy', 'Spain', 'Switzerland']:
        return 'Western Europe'
    elif country in ['Ukraine', 'Russian Federation', 'Romania', 'Bulgaria']:
        return 'Eastern Europe'
    elif country in ['India', 'Pakistan', 'Bangladesh']:
        return 'South Asia'
    elif country in ['China', 'Japan', 'South Korea', 'Singapore', 'Vietnam', 'Indonesia']:
        return 'East/SE Asia'
    elif country in ['Brazil', 'Argentina', 'Mexico', 'Colombia']:
        return 'Latin America'
    elif country in ['Australia', 'New Zealand']:
        return 'Oceania'
    else:
        return 'Other'

df['Region'] = df['Country'].apply(get_region)

print("Regions:")
print(df['Region'].value_counts())

## 13. Create AI Perception Score

In [ ]:
# Composite perception score
df['AI_Perception_Score'] = (
    df['Sentiment_Score'].fillna(3) + 
    df['Trust_Score'].fillna(3) + 
    df['Complexity_Score'].fillna(3)
) / 3

df['AI_Perception_Category'] = pd.cut(
    df['AI_Perception_Score'], 
    bins=[0, 2, 3, 4, 5], 
    labels=['Skeptical', 'Neutral', 'Positive', 'Enthusiastic']
)

print("AI Perception Categories:")
print(df['AI_Perception_Category'].value_counts())

## 14. Threat Perception

In [ ]:
df['Threat_Category'] = df['AIThreat'].map({
    'No': 'Not Threatened',
    'Yes': 'Threatened',
    "I'm not sure": 'Uncertain'
})

print("Threat Perception:")
print(df['Threat_Category'].value_counts())

## 15. Save Cleaned Data

In [ ]:
# Save cleaned data
OUTPUT_PATH = '../data/processed/genai_roi_clean.csv'
df.to_csv(OUTPUT_PATH, index=False)

print(f"\n{'='*50}")
print("✅ DATA CLEANING COMPLETE!")
print(f"{'='*50}")
print(f"\nFinal dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Saved to: {OUTPUT_PATH}")
print(f"\nColumns: {list(df.columns)}")